# Análisis de Resultados MLP

Este notebook analiza los resultados de los experimentos realizados con modelos MLP, comparando diferentes funciones de pérdida (BCE vs Focal Loss) e hiperparámetros.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

sns.set_theme(style="whitegrid")

## Carga de Datos

In [ ]:
# Definir los archivos a cargar
files = [
    "results/mlp_bce.csv",
    "results/mlp_focal.csv",
    "results/mlp_generico.csv"
]

# Buscar el archivo más reciente de resultados refinados
refined_files = glob.glob("results/mlp_focal_refined_results_*.csv")
if refined_files:
    # Tomar el último (asumiendo orden alfabético por timestamp)
    files.append(sorted(refined_files)[-1])

dfs = []
for f in files:
    if os.path.exists(f):
        df = pd.read_csv(f)
        df['Source'] = os.path.basename(f)
        # Normalizar columnas si es necesario
        if 'Loss' not in df.columns:
            if 'generico' in f or 'bce' in f:
                df['Loss'] = 'BCE'
            elif 'focal' in f:
                df['Loss'] = 'FocalLoss'
            else:
                df['Loss'] = 'Unknown'
        
        dfs.append(df)

results = pd.concat(dfs, ignore_index=True)
print(f"Total de experimentos cargados: {len(results)}")
results.head()

## Top 10 Modelos (Micro F1)

In [ ]:
top_10 = results.sort_values('Micro F1', ascending=False).head(10)
top_10[['Model', 'Window Size', 'LR', 'Loss', 'Threshold', 'Micro F1', 'Source']]

## Análisis Visual

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=top_10, x='Micro F1', y='Model', hue='Loss', dodge=False)
plt.title('Top 10 Modelos por Micro F1')
plt.xlabel('Micro F1 (%)')
plt.show()

### Impacto del Window Size

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=results, x='Window Size', y='Micro F1', hue='Loss')
plt.title('Distribución de Micro F1 por Window Size')
plt.show()

### Impacto del Threshold (Solo Focal Loss)

In [ ]:
focal_results = results[results['Loss'] == 'FocalLoss']

if not focal_results.empty:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=focal_results, x='Threshold', y='Micro F1')
    plt.title('Impacto del Threshold en Focal Loss')
    plt.show()

### Impacto de Alpha y Gamma (Solo Focal Loss Refinado)

In [ ]:
# Filtrar resultados que tienen Alpha y Gamma definidos
refined = results.dropna(subset=['Alpha', 'Gamma'])

if not refined.empty:
    pivot_table = refined.pivot_table(values='Micro F1', index='Alpha', columns='Gamma', aggfunc='max')
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(pivot_table, annot=True, fmt=".2f", cmap="viridis")
    plt.title('Max Micro F1 por Alpha y Gamma')
    plt.show()